# Reproducible AI-Assisted Nuclei Segmentation

**Public portfolio project for Manh Tin Ho**

This notebook demonstrates an end-to-end bioimage analysis workflow using the **BBBC039 U2OS nuclei dataset** and **Cellpose-SAM**. It is designed to show the skills that modern wet-lab scientist, imaging scientist, and translational R&D roles increasingly request: public-data access, image quality control, AI model application, model validation, error analysis, reproducibility, and clear scientific reporting.

> Confidentiality note: this public notebook does not contain unpublished cyst-assay data from the University of Bern. It reproduces the same general workflow principles using an open CC0 benchmark dataset.

## Portfolio questions

1. Can a pretrained Cellpose-SAM model segment chemically perturbed U2OS nuclei reliably?
2. Which parameter settings improve validation performance?
3. Does limited fine-tuning improve performance over the pretrained baseline?
4. Where does the model fail, and what biological or imaging conditions explain the errors?

In [ ]:
# Run this cell in Google Colab. Restart the runtime if Colab asks after installation.
%pip -q install "cellpose>=4,<5" tifffile scikit-image pandas matplotlib scipy tqdm pytest

In [ ]:
from pathlib import Path
import os, sys, zipfile, urllib.request, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tifffile import imread
from skimage.measure import regionprops_table, label
from skimage.segmentation import find_boundaries

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "src").exists():
    sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bioimage_portfolio.masks import decode_color_mask
    from bioimage_portfolio.metrics import binary_iou, summarize_counts
except ImportError:
    # Standalone fallback for notebooks opened directly in Google Colab.
    def decode_color_mask(mask):
        arr = np.asarray(mask)
        if arr.ndim == 2:
            unique = np.unique(arr)
            if set(unique.tolist()).issubset({0, 1, 255}):
                return label(arr > 0, connectivity=1).astype(np.int32)
            out = np.zeros(arr.shape, dtype=np.int32)
            for new_label, old_label in enumerate(np.unique(arr[arr != 0]), start=1):
                out[arr == old_label] = new_label
            return out
        rgb = arr[..., :3].astype(np.uint32)
        codes = (rgb[..., 0] << 16) | (rgb[..., 1] << 8) | rgb[..., 2]
        out = np.zeros(codes.shape, dtype=np.int32)
        for new_label, colour in enumerate(np.unique(codes[codes != 0]), start=1):
            out[codes == colour] = new_label
        return out

    def binary_iou(true_mask, pred_mask):
        true_fg = np.asarray(true_mask) > 0
        pred_fg = np.asarray(pred_mask) > 0
        union = np.logical_or(true_fg, pred_fg).sum()
        return 1.0 if union == 0 else float(np.logical_and(true_fg, pred_fg).sum() / union)

    def summarize_counts(true_masks, pred_masks):
        true_counts = np.array([int(np.asarray(m).max()) for m in true_masks], dtype=float)
        pred_counts = np.array([int(np.asarray(m).max()) for m in pred_masks], dtype=float)
        abs_error = np.abs(pred_counts - true_counts)
        denom = np.maximum(true_counts, 1.0)
        return {
            "mean_true_count": float(true_counts.mean()),
            "mean_pred_count": float(pred_counts.mean()),
            "mean_absolute_error": float(abs_error.mean()),
            "mean_absolute_percentage_error": float((abs_error / denom).mean()),
        }

DATA_ROOT = REPO_ROOT / "data" / "BBBC039"
RESULTS_ROOT = REPO_ROOT / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
SEED = 42
np.random.seed(SEED)

## 1. Download the official public dataset

BBBC039 contains 200 Hoechst-stained U2OS fields of view from a chemical screen and approximately 23,000 manually annotated nuclei. The Broad Institute provides official train, validation, and test partitions.

In [ ]:
URLS = {
    "images": "https://data.broadinstitute.org/bbbc/BBBC039/images.zip",
    "masks": "https://data.broadinstitute.org/bbbc/BBBC039/masks.zip",
    "metadata": "https://data.broadinstitute.org/bbbc/BBBC039/metadata.zip",
}

def download_and_extract(name, url):
    archive = DATA_ROOT / f"{name}.zip"
    out_dir = DATA_ROOT / name
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    if not archive.exists():
        print(f"Downloading {name}...")
        urllib.request.urlretrieve(url, archive)
    if not out_dir.exists():
        out_dir.mkdir(parents=True)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(out_dir)
    return out_dir

paths = {name: download_and_extract(name, url) for name, url in URLS.items()}
paths

In [ ]:
image_files = sorted(paths["images"].rglob("*.tif")) + sorted(paths["images"].rglob("*.tiff"))
mask_files = sorted(paths["masks"].rglob("*.png"))
print(f"Images: {len(image_files)} | Masks: {len(mask_files)}")

image_by_stem = {p.stem: p for p in image_files}
mask_by_stem = {p.stem: p for p in mask_files}
common_stems = sorted(set(image_by_stem) & set(mask_by_stem))
print(f"Paired image-mask files: {len(common_stems)}")
assert common_stems, "No image-mask pairs found. Inspect the extracted directory structure."

## 2. Decode masks and perform data quality control

In [ ]:
def load_pair(stem):
    image = imread(image_by_stem[stem])
    raw_mask = plt.imread(mask_by_stem[stem])
    # plt.imread may scale PNG values to 0-1 floats; convert back to uint8.
    if np.issubdtype(raw_mask.dtype, np.floating):
        raw_mask = np.clip(raw_mask * 255, 0, 255).astype(np.uint8)
    mask = decode_color_mask(raw_mask)
    return image, mask

example_stems = common_stems[:4]
fig, axes = plt.subplots(len(example_stems), 3, figsize=(12, 3*len(example_stems)))
for row, stem in enumerate(example_stems):
    image, mask = load_pair(stem)
    boundaries = find_boundaries(mask)
    overlay = np.stack([image, image, image], axis=-1).astype(float)
    overlay = (overlay - overlay.min()) / max(overlay.max() - overlay.min(), 1)
    overlay[boundaries] = [1, 0, 0]
    axes[row, 0].imshow(image, cmap="gray")
    axes[row, 0].set_title(stem)
    axes[row, 1].imshow(mask, cmap="nipy_spectral")
    axes[row, 1].set_title(f"Ground truth: {mask.max()} nuclei")
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title("Boundary overlay")
    for ax in axes[row]: ax.axis("off")
plt.tight_layout()

In [ ]:
qc_rows = []
for stem in common_stems:
    image, mask = load_pair(stem)
    qc_rows.append({
        "stem": stem,
        "height": image.shape[0],
        "width": image.shape[1],
        "dtype": str(image.dtype),
        "p01": float(np.percentile(image, 1)),
        "p99": float(np.percentile(image, 99)),
        "n_nuclei": int(mask.max()),
        "foreground_fraction": float((mask > 0).mean()),
    })
qc = pd.DataFrame(qc_rows)
qc.describe(include="all").T

## 3. Create official or deterministic data splits

The notebook first tries to read the Broad Institute metadata. If split files cannot be identified automatically, it falls back to a deterministic 70/15/15 split and clearly reports that fallback.

In [ ]:
def read_split_metadata(meta_root, available_stems):
    result = {}
    for file in meta_root.rglob("*"):
        if not file.is_file() or file.suffix.lower() not in {".txt", ".csv", ".tsv"}:
            continue
        lower = file.name.lower()
        split = next((s for s in ["train", "validation", "valid", "test"] if s in lower), None)
        if split is None:
            continue
        text = file.read_text(errors="ignore")
        names = set()
        for token in text.replace(",", " ").split():
            stem = Path(token.strip()).stem
            if stem in available_stems:
                names.add(stem)
        if names:
            key = "validation" if split == "valid" else split
            result.setdefault(key, set()).update(names)
    return {k: sorted(v) for k, v in result.items()}

splits = read_split_metadata(paths["metadata"], set(common_stems))
if not {"train", "validation", "test"}.issubset(splits):
    print("Official split files were not fully detected; using deterministic fallback split.")
    rng = np.random.default_rng(SEED)
    shuffled = np.array(common_stems)
    rng.shuffle(shuffled)
    n = len(shuffled)
    splits = {
        "train": shuffled[:int(0.70*n)].tolist(),
        "validation": shuffled[int(0.70*n):int(0.85*n)].tolist(),
        "test": shuffled[int(0.85*n):].tolist(),
    }
{k: len(v) for k, v in splits.items()}

## 4. Pretrained Cellpose-SAM baseline

In [ ]:
import torch
from cellpose import models, metrics

USE_GPU = torch.cuda.is_available()
print("GPU available:", USE_GPU)
model = models.CellposeModel(gpu=USE_GPU, pretrained_model="cpsam_v2")

MAX_VAL_IMAGES = 20  # Increase for a fuller benchmark.
val_stems = splits["validation"][:MAX_VAL_IMAGES]
val_images, val_true = zip(*(load_pair(stem) for stem in val_stems))

start = time.perf_counter()
val_pred, val_flows, _ = model.eval(list(val_images), flow_threshold=0.4, cellprob_threshold=0.0)
elapsed = time.perf_counter() - start
print(f"Segmented {len(val_images)} images in {elapsed:.1f} s")

In [ ]:
def evaluate_instance_masks(true_masks, pred_masks):
    ap, tp, fp, fn = metrics.average_precision(true_masks, pred_masks, threshold=[0.5, 0.75, 0.9])
    aji = metrics.aggregated_jaccard_index(true_masks, pred_masks)
    count_summary = summarize_counts(true_masks, pred_masks)
    return {
        "AP@0.50": float(np.mean(ap[:, 0])),
        "AP@0.75": float(np.mean(ap[:, 1])),
        "AP@0.90": float(np.mean(ap[:, 2])),
        "AJI": float(np.mean(aji)),
        **count_summary,
    }

baseline_metrics = evaluate_instance_masks(list(val_true), list(val_pred))
pd.Series(baseline_metrics, name="pretrained_baseline")

## 5. Parameter sweep and validation-driven model selection

A scientist should not simply accept default AI output. This section demonstrates structured parameter testing on a validation set, without using the held-out test set for model selection.

In [ ]:
parameter_grid = [
    {"flow_threshold": 0.2, "cellprob_threshold": -1.0},
    {"flow_threshold": 0.4, "cellprob_threshold": -1.0},
    {"flow_threshold": 0.4, "cellprob_threshold": 0.0},
    {"flow_threshold": 0.6, "cellprob_threshold": 0.0},
    {"flow_threshold": 0.4, "cellprob_threshold": 1.0},
]

rows = []
for params in parameter_grid:
    pred, _, _ = model.eval(list(val_images), **params)
    result = evaluate_instance_masks(list(val_true), list(pred))
    rows.append({**params, **result})
parameter_results = pd.DataFrame(rows).sort_values("AP@0.50", ascending=False)
parameter_results

## 6. Limited fine-tuning on manually annotated data

The official Cellpose guidance recommends training from a built-in model and keeping all images intended for the same model in the training folder. This demonstration limits the number of images so it can run in Colab; increase the constants for a fuller study.

In [ ]:
from cellpose import train

MAX_TRAIN_IMAGES = 40
MAX_TRAIN_EPOCHS = 100
train_stems = splits["train"][:MAX_TRAIN_IMAGES]
train_pairs = [load_pair(stem) for stem in train_stems]
train_images = [p[0] for p in train_pairs]
train_labels = [p[1] for p in train_pairs]

finetune_model = models.CellposeModel(gpu=USE_GPU, pretrained_model="cpsam_v2")
model_path, train_losses, test_losses = train.train_seg(
    finetune_model.net,
    train_data=train_images,
    train_labels=train_labels,
    test_data=list(val_images),
    test_labels=list(val_true),
    learning_rate=1e-5,
    weight_decay=0.1,
    n_epochs=MAX_TRAIN_EPOCHS,
    batch_size=1,
    model_name="bbbc039_u2os_finetuned",
    save_path=str(RESULTS_ROOT),
)
print("Saved fine-tuned model:", model_path)

In [ ]:
fine_model = models.CellposeModel(gpu=USE_GPU, pretrained_model=str(model_path))
fine_pred, fine_flows, _ = fine_model.eval(list(val_images))
fine_metrics = evaluate_instance_masks(list(val_true), list(fine_pred))
pd.DataFrame([baseline_metrics, fine_metrics], index=["pretrained", "fine_tuned"])

## 7. Error analysis and biological interpretation

In [ ]:
comparison = []
for stem, image, true_mask, pred_mask in zip(val_stems, val_images, val_true, fine_pred):
    comparison.append({
        "stem": stem,
        "true_count": int(true_mask.max()),
        "pred_count": int(pred_mask.max()),
        "count_error": int(pred_mask.max() - true_mask.max()),
        "foreground_iou": binary_iou(true_mask, pred_mask),
        "density": float((true_mask > 0).mean()),
    })
error_df = pd.DataFrame(comparison).sort_values("foreground_iou")
error_df.head(10)

In [ ]:
worst = error_df.head(4)["stem"].tolist()
fig, axes = plt.subplots(len(worst), 3, figsize=(12, 3*len(worst)))
for row, stem in enumerate(worst):
    idx = val_stems.index(stem)
    image, true_mask, pred_mask = val_images[idx], val_true[idx], fine_pred[idx]
    axes[row, 0].imshow(image, cmap="gray")
    axes[row, 0].set_title(stem)
    axes[row, 1].imshow(true_mask, cmap="nipy_spectral")
    axes[row, 1].set_title(f"Ground truth ({true_mask.max()})")
    axes[row, 2].imshow(pred_mask, cmap="nipy_spectral")
    axes[row, 2].set_title(f"Prediction ({pred_mask.max()})")
    for ax in axes[row]: ax.axis("off")
plt.tight_layout()

## 8. Export a reproducible result package

In [ ]:
summary = {
    "dataset": "BBBC039v1",
    "model": "Cellpose-SAM cpsam_v2",
    "seed": SEED,
    "validation_images": len(val_images),
    "baseline_metrics": baseline_metrics,
    "fine_tuned_metrics": fine_metrics,
}
(RESULTS_ROOT / "benchmark_summary.json").write_text(json.dumps(summary, indent=2))
parameter_results.to_csv(RESULTS_ROOT / "parameter_sweep.csv", index=False)
error_df.to_csv(RESULTS_ROOT / "error_analysis.csv", index=False)
qc.to_csv(RESULTS_ROOT / "dataset_qc.csv", index=False)
print("Saved:", sorted(p.name for p in RESULTS_ROOT.iterdir()))

## CV-ready evidence statement

After completing and publishing the benchmark, use a statement such as:

> Built a reproducible Cellpose-SAM benchmark on 200 chemically perturbed U2OS microscopy images with approximately 23,000 annotated nuclei; performed quality control, validation-based parameter selection, limited fine-tuning, AP/AJI evaluation, and structured error analysis in Python.

Do not insert numerical performance claims until the notebook has been run and the result files have been reviewed.